03.4 Automated ETL Pipeline: Build labeled_step_test

In [0]:
### Load raw tables, normalize columns, label step events, and build final_df ###

from pyspark.sql.functions import col, lit, when, to_timestamp, regexp_extract

# Load from bronze (adjust if your raw tables are elsewhere)
df_device = spark.table("workspace.bronze.device_messages_raw")
df_steps  = spark.table("workspace.bronze.rapid_step_tests_raw")

def rename_if_exists(df, old, new):
    return df.withColumnRenamed(old, new) if old in df.columns else df

# Normalize device table
d = df_device
d = rename_if_exists(d, "deviceId", "device_id")
d = rename_if_exists(d, "sensorType", "sensor_type")
d = d.withColumn("timestamp", to_timestamp(col("timestamp")))

from pyspark.sql.functions import regexp_extract, when, length, lit, col

# extract digits from things like "1cm", "12 cm", etc.
dist_digits = regexp_extract(col("distance"), r"(\d+)", 1)

d = d.withColumn(
    "distance_cm",
    when(length(dist_digits) > 0, dist_digits.cast("int")).otherwise(lit(None).cast("int"))
)



# Normalize steps table
s = df_steps
s = rename_if_exists(s, "deviceId", "device_id")
s = rename_if_exists(s, "startTime", "start_time")
s = rename_if_exists(s, "stopTime", "stop_time")
s = s.withColumn("start_time", to_timestamp(col("start_time")))
s = s.withColumn("stop_time",  to_timestamp(col("stop_time")))

# Build step windows with non-colliding key name
s_win = (
    s.select("device_id", "start_time", "stop_time")
     .dropna(subset=["device_id", "start_time", "stop_time"])
     .withColumnRenamed("device_id", "step_device_id")
)

# Join + label
labeled = (
    d.alias("d")
     .join(
         s_win.alias("s"),
         (col("d.device_id") == col("s.step_device_id")) &
         (col("d.timestamp").between(col("s.start_time"), col("s.stop_time"))),
         "left"
     )
     .withColumn(
         "step_label",
         when(col("s.start_time").isNotNull(), lit("step")).otherwise(lit("no_step"))
     )
     .withColumn("source_label", lit("device"))
)

# Final curated dataframe expected by the assignment queries
final_df = labeled.select(
    "timestamp",
    "sensor_type",
    "distance_cm",
    "device_id",
    "step_label",
    "source_label"
)

# Make it available to SQL as final_df (their instructions require this pattern)
final_df.createOrReplaceTempView("final_df")


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.labeled_step_test AS
SELECT * FROM final_df;


# Verification

In [0]:
%sql
SELECT step_label, COUNT(*)
FROM workspace.silver.labeled_step_test
GROUP BY step_label;


In [0]:
%sql
SELECT *
FROM workspace.silver.labeled_step_test
WHERE step_label NOT IN ('step','no_step')
   OR step_label IS NULL
LIMIT 50;


In [0]:
%sql
SELECT source_label, COUNT(*)
FROM workspace.silver.labeled_step_test
GROUP BY source_label;


In [0]:
%sql
SELECT *
FROM workspace.silver.labeled_step_test
WHERE source_label NOT IN ('device','step')
   OR source_label IS NULL
LIMIT 50;
